## Setup Instructions:

### Before running this Notebook run these commands in the codespace terminal:

python -m pip install ipykernel 
- (may require a page refresh before kernel selection)

sudo apt-get update  

sudo DEBIAN_FRONTEND=noninteractive apt-get install -y tshark  


### Do this as well:

Click 'Select Kernel' in the top right of this notebook  

Install Jupyter/Python .venv kernel  

Select the .venv kernel (it should have a star next to it)

### Warning:

This notebook will save in the codespace you have booted. If you delete the codespace without commiting to git, you will lose work. 

In [1]:
pip install pyshark pandas matplotlib nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [2]:
import nest_asyncio
nest_asyncio.apply()

import pyshark
import pandas as pd
from datetime import datetime

# Full DNS RR type map (numeric -> name)
DNS_TYPES = {
    1: 'A', 2: 'NS', 5: 'CNAME', 6: 'SOA', 12: 'PTR', 15: 'MX', 16: 'TXT',
    28: 'AAAA', 33: 'SRV', 41: 'OPT', 43: 'DS', 46: 'RRSIG', 47: 'NSEC',
    48: 'DNSKEY', 50: 'NSEC3', 51: 'NSEC3PARAM', 52: 'TLSA', 257: 'CAA',
    255: 'ANY', 6: 'SOA', 25: 'KEY'
}

# DNSSEC signing algorithm numbers -> names
DNSSEC_ALGOS = {
    1: 'RSA/MD5', 3: 'DSA/SHA1', 5: 'RSA/SHA-1',
    6: 'DSA-NSEC3-SHA1', 7: 'RSASHA1-NSEC3-SHA1',
    8: 'RSA/SHA-256', 10: 'RSA/SHA-512',
    13: 'ECDSA P-256/SHA-256', 14: 'ECDSA P-384/SHA-384',
    15: 'Ed25519', 16: 'Ed448'
}

DS_DIGEST_TYPES = {1: 'SHA-1', 2: 'SHA-256', 3: 'GOST R 34.11-94', 4: 'SHA-384'}

def type_name(val):
    try:
        return DNS_TYPES.get(int(val), f'TYPE{val}')
    except (ValueError, TypeError):
        return str(val)

def algo_name(val):
    try:
        return DNSSEC_ALGOS.get(int(val), f'ALG{val}')
    except (ValueError, TypeError):
        return str(val)

def digest_name(val):
    try:
        return DS_DIGEST_TYPES.get(int(val), f'DIGEST{val}')
    except (ValueError, TypeError):
        return str(val)

def parse_dnssec_time(val):
    """tshark gives RRSIG times either as epoch seconds or 'YYYYMMDDHHMMSS' string."""
    if val is None:
        return '?'
    s = str(val)
    try:
        # Sometimes formatted like "Feb 12, 2013 00:00:00 UTC" already - just return it
        if any(c.isalpha() for c in s):
            return s
        if len(s) == 14 and s.isdigit():
            dt = datetime.strptime(s, '%Y%m%d%H%M%S')
            return dt.strftime('%Y-%m-%d %H:%M:%S UTC')
        # otherwise treat as epoch
        return datetime.utcfromtimestamp(int(s)).strftime('%Y-%m-%d %H:%M:%S UTC')
    except Exception:
        return s

In [3]:
def analyze_pcap(pcap_path, display_filter='dns'):
    cap = pyshark.FileCapture(pcap_path, display_filter=display_filter)
    for i, packet in enumerate(cap):
        dns = packet.dns
        print(f"\n--- Packet {i} | {packet.sniff_time} ---")

        is_response = getattr(dns, 'flags_response', '0') == '1'
        print(f"Type: {'Response' if is_response else 'Query'}")

        if hasattr(dns, 'qry_name'):
            print(f"  Query name: {dns.qry_name}")
        if hasattr(dns, 'qry_type'):
            print(f"  Query type: {type_name(dns.qry_type)}")

        if hasattr(dns, 'flags_authenticated'):
            print(f"  Flags: AD={getattr(dns, 'flags_authenticated', '?')} "
                f"CD={getattr(dns, 'flags_checkdisable', '?')}")

        do_bit = getattr(dns, 'resp_z_do', None) or getattr(dns, 'qry_z_do', None)
        if do_bit is not None:
            print(f"  DO bit (DNSSEC requested): {do_bit}")

        if hasattr(dns, 'resp_type'):
            resp_types = dns.resp_type.all_fields if hasattr(dns.resp_type, 'all_fields') else [dns.resp_type]
            for rt in resp_types:
                rt_val = str(rt.show if hasattr(rt, 'show') else rt)
                print(f"  Response record type: {type_name(rt_val)}")

        if hasattr(dns, 'rrsig_type_covered'):
            print(f"  RRSIG covers type: {type_name(dns.rrsig_type_covered)}")
            print(f"  RRSIG algorithm: {algo_name(getattr(dns, 'rrsig_algorithm', None))}")
            print(f"  RRSIG signer name: {getattr(dns, 'rrsig_signers_name', '?')}")
            print(f"  RRSIG expiration: {getattr(dns, 'rrsig_signature_expiration', '?')}")
            print(f"  RRSIG inception: {getattr(dns, 'rrsig_signature_inception', '?')}")

        if hasattr(dns, 'dnskey_flags'):
            flags_val = int(str(dns.dnskey_flags), 0)
            flag_desc = []
            if flags_val & 0x0001:
                flag_desc.append('Zone Key')
            if flags_val & 0x0100:
                flag_desc.append('ZSK')
            if flags_val & 0x0101 == 0x0101:
                flag_desc = ['KSK/SEP + Zone Key']
            print(f"  DNSKEY flags: {dns.dnskey_flags} ({', '.join(flag_desc) if flag_desc else 'none'})")
            print(f"  DNSKEY algorithm: {algo_name(getattr(dns, 'dnskey_algorithm', None))}")
            print(f"  DNSKEY key id: {getattr(dns, 'dnskey_key_id', '?')}")

        if hasattr(dns, 'ds_key_id'):
            print(f"  DS key tag: {dns.ds_key_id}")
            print(f"  DS digest type: {digest_name(getattr(dns, 'ds_digest_type', None))}")

        if hasattr(dns, 'nsec_next_domain_name'):
            print(f"  NSEC next domain: {dns.nsec_next_domain_name}")
        if hasattr(dns, 'nsec3_hash_algorithm'):
            print(f"  NSEC3 hash algorithm: {dns.nsec3_hash_algorithm}")
        if hasattr(dns, 'flags_rcode'):
            rcode_map = {0: 'NOERROR', 2: 'SERVFAIL', 3: 'NXDOMAIN', 5: 'REFUSED'}
            rc = int(str(dns.flags_rcode), 0)
            print(f"  RCODE: {rcode_map.get(rc, f'RCODE{rc}')}")

    cap.close()

In [4]:
def summarize_pcap(pcap_path, display_filter='dns'):
    """
    Build a one-row-per-packet summary table for a capture, instead of the
    verbose per-packet printout from analyze_pcap(). Useful for getting a
    quick overview of an entire chain (query name, type, flags, RRSIG signer,
    RCODE) without scrolling through long text blocks.
    Returns a pandas DataFrame.
    """
    cap = pyshark.FileCapture(pcap_path, display_filter=display_filter)
    rows = []
    try:
        for i, packet in enumerate(cap):
            dns = packet.dns

            is_response = getattr(dns, 'flags_response', '0') == '1'

            query_name = getattr(dns, 'qry_name', None)
            query_type = type_name(dns.qry_type) if hasattr(dns, 'qry_type') else None

            ad_flag = getattr(dns, 'flags_authenticated', None)
            do_bit = getattr(dns, 'resp_z_do', None) or getattr(dns, 'qry_z_do', None)

            rrsig_signer = getattr(dns, 'rrsig_signers_name', None)

            rcode = None
            if hasattr(dns, 'flags_rcode'):
                rcode_map = {0: 'NOERROR', 2: 'SERVFAIL', 3: 'NXDOMAIN', 5: 'REFUSED'}
                rc = int(str(dns.flags_rcode), 0)
                rcode = rcode_map.get(rc, f'RCODE{rc}')

            rows.append({
                'packet': i,
                'time': packet.sniff_time,
                'type': 'Response' if is_response else 'Query',
                'query_name': query_name,
                'query_type': query_type,
                'AD': ad_flag,
                'DO': do_bit,
                'RRSIG_signer': rrsig_signer,
                'RCODE': rcode,
            })
    finally:
        cap.close()

    return pd.DataFrame(rows)

In [5]:
analyze_pcap('./dnssec.pcap')


--- Packet 0 | 2013-01-23 02:48:50.954626 ---
Type: Query
  Query name: www.ietf.org
  Query type: A
  DO bit (DNSSEC requested): 1
  Response record type: OPT

--- Packet 1 | 2013-01-23 02:48:51.029064 ---
Type: Response
  Query name: www.ietf.org
  Query type: A
  Flags: AD=1 CD=0
  DO bit (DNSSEC requested): 1
  Response record type: A
  Response record type: RRSIG
  Response record type: OPT
  RRSIG covers type: A
  RRSIG algorithm: RSA/SHA-1
  RRSIG signer name: ietf.org
  RRSIG expiration: Jan 10, 2014 22:18:47.000000000 UTC
  RRSIG inception: Jan 10, 2013 21:20:47.000000000 UTC
  RCODE: NOERROR

--- Packet 2 | 2013-01-23 02:48:51.029916 ---
Type: Query
  Query name: ietf.org
  Query type: DNSKEY
  DO bit (DNSSEC requested): 1
  Response record type: OPT

--- Packet 3 | 2013-01-23 02:48:51.102520 ---
Type: Response
  Query name: ietf.org
  Query type: DNSKEY
  Flags: AD=1 CD=0
  DO bit (DNSSEC requested): 1
  Response record type: DNSKEY
  Response record type: DNSKEY
  Response

In [6]:
summarize_pcap('./dnssec.pcap')

,packet,time,type,query_name,query_type,AD,DO,RRSIG_signer,RCODE
0,0,2013-01-23 02:48:50.954626,Query,www.ietf.org,A,NaN,1,NaN,NaN
1,1,2013-01-23 02:48:51.029064,Response,www.ietf.org,A,1,1,ietf.org,NOERROR
2,2,2013-01-23 02:48:51.029916,Query,ietf.org,DNSKEY,NaN,1,NaN,NaN
3,3,2013-01-23 02:48:51.102520,Response,ietf.org,DNSKEY,1,1,ietf.org,NOERROR
4,4,2013-01-23 02:48:51.102809,Query,ietf.org,DS,NaN,1,NaN,NaN
5,5,2013-01-23 02:48:51.171976,Response,ietf.org,DS,1,1,org,NOERROR
6,6,2013-01-23 02:48:51.172929,Query,org,DNSKEY,NaN,1,NaN,NaN
7,7,2013-01-23 02:48:51.243571,Response,org,DNSKEY,1,1,org,NOERROR
8,8,2013-01-23 02:48:51.243848,Query,org,DS,NaN,1,NaN,NaN
9,9,2013-01-23 02:48:51.313142,Response,org,DS,1,1,<Root>,NOERROR


### PCAP Analysis #1:

#### Using the DNSSEC capture above, answer these questions:

1) Trace the chain of trust from www.ietf.org back to the root. At each hop, which key signs which record, and how does the next query know which key to trust?  

2) Packet 9's RRSIG signer is <Root> but packet 5's is org, and packet 3's is ietf.org. What dictates who signs each packet?

3) Why does packet 13 have NSEC records while no other packets do?  

In [7]:
analyze_pcap('./calvin.pcapng', 'dns and ip.addr == 8.8.8.8')


--- Packet 0 | 2026-07-23 13:29:32.606349 ---
Type: Query
  Query name: www.calvin.edu
  Query type: A
  DO bit (DNSSEC requested): 1
  Response record type: OPT

--- Packet 1 | 2026-07-23 13:29:32.646358 ---
Type: Response
  Query name: www.calvin.edu
  Query type: A
  Flags: AD=0 CD=0
  DO bit (DNSSEC requested): 1
  Response record type: A
  Response record type: OPT
  RCODE: NOERROR

--- Packet 2 | 2026-07-23 13:29:32.646346 ---
Type: Query
  Query name: edu
  Query type: DS
  DO bit (DNSSEC requested): 1
  Response record type: OPT

--- Packet 3 | 2026-07-23 13:29:32.658684 ---
Type: Response
  Query name: edu
  Query type: DS
  Flags: AD=1 CD=0
  DO bit (DNSSEC requested): 1
  Response record type: DS
  Response record type: RRSIG
  Response record type: OPT
  RRSIG covers type: DS
  RRSIG algorithm: RSA/SHA-256
  RRSIG signer name: <Root>
  RRSIG expiration: Aug  5, 2026 05:00:00.000000000 UTC
  RRSIG inception: Jul 23, 2026 04:00:00.000000000 UTC
  DS key tag: 0x8b4f
  DS dige

In [8]:
summarize_pcap('./calvin.pcapng', 'dns and ip.addr == 8.8.8.8')

,packet,time,type,query_name,query_type,AD,DO,RRSIG_signer,RCODE
0,0,2026-07-23 13:29:32.606349,Query,www.calvin.edu,A,NaN,1,NaN,NaN
1,1,2026-07-23 13:29:32.646358,Response,www.calvin.edu,A,0,1,NaN,NOERROR
2,2,2026-07-23 13:29:32.646346,Query,edu,DS,NaN,1,NaN,NaN
3,3,2026-07-23 13:29:32.658684,Response,edu,DS,1,1,<Root>,NOERROR
4,4,2026-07-23 13:29:32.646346,Query,<Root>,DNSKEY,NaN,1,NaN,NaN
5,5,2026-07-23 13:29:32.669912,Response,<Root>,DNSKEY,1,1,<Root>,NOERROR
6,6,2026-07-23 13:29:32.669344,Query,calvin.edu,DS,NaN,1,NaN,NaN
7,7,2026-07-23 13:29:32.705016,Response,calvin.edu,DS,0,1,edu,NOERROR
8,8,2026-07-23 13:29:32.704341,Query,edu,DNSKEY,NaN,1,NaN,NaN
9,9,2026-07-23 13:29:32.717097,Response,edu,DNSKEY,1,1,edu,NOERROR


### PCAP Analysis #2:

#### Using the DNSSEC capture above, answer these questions:

1) In packet 1 of this capture, the AD (Authentic Data) flag = 0, but every packet resolving the chain has AD=1. If the whle chain is validated, why is the actual answer unauthenticated?

2) This capture never shows a query/response for calvin.edu's own DNSKEY, only its DS record. Why is this?

In [9]:
analyze_pcap('./colvin.pcapng', 'dns and ip.addr == 8.8.8.8')



--- Packet 0 | 2026-07-23 13:33:38.187488 ---
Type: Query
  Query name: www.colvin.edu
  Query type: A
  DO bit (DNSSEC requested): 1
  Response record type: OPT

--- Packet 1 | 2026-07-23 13:33:38.228312 ---
Type: Response
  Query name: www.colvin.edu
  Query type: A
  Flags: AD=0 CD=0
  DO bit (DNSSEC requested): 1
  Response record type: SOA
  Response record type: RRSIG
  Response record type: NSEC3
  Response record type: NS
  Response record type: SOA
  Response record type: RRSIG
  Response record type: DNSKEY
  Response record type: NSEC3PARAM
  Response record type: RRSIG
  Response record type: NSEC3
  Response record type: NS
  Response record type: DS
  Response record type: RRSIG
  Response record type: RRSIG
  Response record type: NSEC3
  Response record type: NS
  Response record type: DS
  Response record type: RRSIG
  Response record type: RRSIG
  Response record type: OPT
  RRSIG covers type: SOA
  RRSIG algorithm: ECDSA P-256/SHA-256
  RRSIG signer name: edu
  RRSI

In [10]:
summarize_pcap('./colvin.pcapng', 'dns and ip.addr == 8.8.8.8')

,packet,time,type,query_name,query_type,AD,DO,RRSIG_signer,RCODE
0,0,2026-07-23 13:33:38.187488,Query,www.colvin.edu,A,NaN,1,NaN,NaN
1,1,2026-07-23 13:33:38.228312,Response,www.colvin.edu,A,0,1,edu,NXDOMAIN
2,2,2026-07-23 13:33:38.229020,Query,edu,DNSKEY,NaN,1,NaN,NaN
3,3,2026-07-23 13:33:38.250187,Response,edu,DNSKEY,1,1,edu,NOERROR
4,4,2026-07-23 13:33:38.250421,Query,edu,DS,NaN,1,NaN,NaN
5,5,2026-07-23 13:33:38.271619,Response,edu,DS,1,1,<Root>,NOERROR
6,6,2026-07-23 13:33:38.266856,Query,<Root>,DNSKEY,NaN,1,NaN,NaN
7,7,2026-07-23 13:33:38.296133,Response,<Root>,DNSKEY,1,1,<Root>,NOERROR


### PCAP Analysis #3:

#### Using the DNSSEC capture above, answer these questions:

1) Packet 1 returns RCODE: NXDOMAIN, yet it's carrying NSEC3, RRSIG, DS, and DNSKEY records. If the domain doesn't exist, what are all of these records proving, and to whom?

2) This query is a spelling error from calvin.edu to colvin.edu, why does it matter than NSEC3 and not NSEC was used here?

In [11]:
analyze_pcap('./failed.pcapng', 'dns and ip.addr == 8.8.8.8')


--- Packet 0 | 2026-07-23 13:35:51.749954 ---
Type: Query
  Query name: www.dnssec-failed.org
  Query type: A
  DO bit (DNSSEC requested): 1
  Response record type: OPT

--- Packet 1 | 2026-07-23 13:35:51.953508 ---
Type: Response
  Query name: www.dnssec-failed.org
  Query type: A
  Flags: AD=0 CD=0
  DO bit (DNSSEC requested): 1
  Response record type: OPT
  RCODE: SERVFAIL


In [12]:
summarize_pcap('./failed.pcapng', 'dns and ip.addr == 8.8.8.8')

,packet,time,type,query_name,query_type,AD,DO,RRSIG_signer,RCODE
0,0,2026-07-23 13:35:51.749954,Query,www.dnssec-failed.org,A,NaN,1,None,NaN
1,1,2026-07-23 13:35:51.953508,Response,www.dnssec-failed.org,A,0,1,None,SERVFAIL


### PCAP Analysis #4:

#### Using the DNSSEC capture above, answer these questions:

1) What does the difference between this packet, SERVFAIL, and packet #3 NXDOMAIN tell us?